# Notebook 06 — Comparative Evaluation
Loads all three trained models, produces the final comparison table, ROC overlay, and all figures.

In [ ]:
# Pull latest code from GitHub
import subprocess
result = subprocess.run(['git', 'pull'], capture_output=True, text=True, cwd='..')
print(result.stdout or 'Already up to date.')
if result.returncode != 0:
    print('stderr:', result.stderr)

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

splits_dir  = Path('../' + cfg['data']['splits_path'])
results_dir = Path('../results')
figures_dir = results_dir / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load test splits — LSTM/GRU use seq splits; RF uses flat splits
X_test_seq  = np.load(splits_dir / 'X_test_seq.npy')
y_test_seq  = np.load(splits_dir / 'y_test_seq.npy')
X_test_flat = np.load(splits_dir / 'X_test.npy')
y_test_flat = np.load(splits_dir / 'y_test.npy')

n_features = X_test_seq.shape[2]
print('Seq test:', X_test_seq.shape, '  Flat test:', X_test_flat.shape)

## Load Models

In [ ]:
from src.models.lstm_model import build_lstm
from src.models.gru_model  import build_gru
from src.models.rf_model   import load_rf

lstm = build_lstm(cfg, n_features).to(device)
lstm.load_state_dict(torch.load('../results/checkpoints/best_lstm.pt', map_location=device))
lstm.eval()

gru = build_gru(cfg, n_features).to(device)
gru.load_state_dict(torch.load('../results/checkpoints/best_gru.pt', map_location=device))
gru.eval()

rf = load_rf('../results/checkpoints/best_rf.pkl')

print('All models loaded.')

## Per-Model Evaluation

In [ ]:
from src.evaluate import evaluate_dl_model, evaluate_rf_model
from sklearn.metrics import roc_curve, auc

_, y_prob_lstm = evaluate_dl_model(
    lstm, X_test_seq, y_test_seq, model_name='LSTM',
    figures_dir='../results/figures', csv_path='../results/metrics_summary.csv')

_, y_prob_gru  = evaluate_dl_model(
    gru, X_test_seq, y_test_seq, model_name='GRU',
    figures_dir='../results/figures', csv_path='../results/metrics_summary.csv')

_, y_prob_rf   = evaluate_rf_model(
    rf, X_test_flat, y_test_flat,
    figures_dir='../results/figures', csv_path='../results/metrics_summary.csv')

## ROC Curve Overlay

In [ ]:
from src.utils import plot_roc_overlay

fpr_lstm, tpr_lstm, _ = roc_curve(y_test_seq,  y_prob_lstm, pos_label=1)
fpr_gru,  tpr_gru,  _ = roc_curve(y_test_seq,  y_prob_gru,  pos_label=1)
fpr_rf,   tpr_rf,   _ = roc_curve(y_test_flat, y_prob_rf,   pos_label=1)

roc_data = {
    'LSTM': (fpr_lstm, tpr_lstm, auc(fpr_lstm, tpr_lstm)),
    'GRU':  (fpr_gru,  tpr_gru,  auc(fpr_gru,  tpr_gru)),
    'RF':   (fpr_rf,   tpr_rf,   auc(fpr_rf,   tpr_rf)),
}

plot_roc_overlay(roc_data, save_path='../results/figures/roc_comparison.png')

## Final Comparison Table

In [ ]:
summary = pd.read_csv('../results/metrics_summary.csv')

# Filter to test-set evaluation rows (model names only, no exp_id prefix)
test_rows = summary[summary['model'].isin(['LSTM', 'GRU', 'RandomForest'])].copy()
# Keep only the final evaluation rows (they have roc_auc filled)
test_rows = test_rows.dropna(subset=['roc_auc'])

display_cols = ['model', 'accuracy', 'precision_ddos', 'recall_ddos', 'f1_ddos', 'fpr', 'roc_auc']
final_table = test_rows[display_cols].drop_duplicates(subset='model').set_index('model')
print(final_table.to_string())
final_table.to_csv('../results/final_comparison.csv')
print('\nSaved to results/final_comparison.csv')

## Hyperparameter Tuning Summary

In [ ]:
print('=== LSTM experiments ===')
print(summary[summary['model'] == 'LSTM'][['exp_id', 'hidden_size', 'num_layers',
      'dropout', 'seq_len', 'lr', 'best_val_f1', 'notes']].to_string(index=False))

print('\n=== GRU experiments ===')
print(summary[summary['model'] == 'GRU'][['exp_id', 'hidden_size', 'num_layers',
      'dropout', 'seq_len', 'lr', 'best_val_f1', 'notes']].to_string(index=False))

print('\n=== RF experiments ===')
print(summary[summary['model'] == 'RandomForest'][['exp_id', 'n_estimators',
      'max_depth', 'best_val_f1', 'notes']].to_string(index=False))

## Bar Chart — F1 / AUC Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

models = final_table.index.tolist()
colors = ['#4C72B0', '#55A868', '#DD8452']

axes[0].bar(models, final_table['f1_ddos'], color=colors)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('F1 Score (DDoS class)')
axes[0].set_title('F1 Score Comparison')
for i, v in enumerate(final_table['f1_ddos']):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=9)

axes[1].bar(models, final_table['roc_auc'], color=colors)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('ROC-AUC Comparison')
for i, v in enumerate(final_table['roc_auc']):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=9)

plt.suptitle('Model Comparison — CIC-DDoS2019', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/model_comparison_bar.png', dpi=150)
plt.show()